# 00 Data Loading And Split

            Use this notebook first.
            It is the place to:

            - inspect exported `episode_frames`
            - build the train-ready sample table
            - save one shared train/val/test split file
            - reuse that same split for every model notebook


In [1]:
import json
from pathlib import Path
import sys

PROJECT_ROOT = Path.home() / "Documents/Thesis"
PIPELINE_ROOT = PROJECT_ROOT / "08_model_training_pipeline"
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from datasets.paths import EPISODE_FRAMES_ROOT, SPLITS_ROOT, TRAIN_READY_ROOT
from datasets.sample_table import build_sample_table, load_episode_streams, save_or_load_fixed_split, save_sample_table

SEED = 42
PAST_LEN = 10
FUTURE_LEN = 5
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

print("Pipeline root:", PIPELINE_ROOT)
print("Episode frames root:", EPISODE_FRAMES_ROOT)
print("Split strategy: episode-level shared split")


Pipeline root: /home/basudeo/Documents/Thesis/08_model_training_pipeline
Episode frames root: /home/basudeo/Documents/Thesis/08_model_training_pipeline/results/episode_frames
Split strategy: episode-level shared split


In [2]:
sample_files = sorted(EPISODE_FRAMES_ROOT.glob("*/frames.jsonl"))
print("Episode files:", len(sample_files))
for path in sample_files[:10]:
    print(path)


Episode files: 6
/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/episode_frames/run_model_20260511_202309/frames.jsonl
/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/episode_frames/run_model_20260511_210809/frames.jsonl
/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/episode_frames/run_model_20260511_213251/frames.jsonl
/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/episode_frames/run_model_20260511_220103/frames.jsonl
/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/episode_frames/run_model_20260511_221726/frames.jsonl
/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/episode_frames/run_model_20260511_222947/frames.jsonl


In [3]:
streams = load_episode_streams(EPISODE_FRAMES_ROOT)
sample_table = build_sample_table(streams, past_len=PAST_LEN, future_len=FUTURE_LEN)

sample_table_path = TRAIN_READY_ROOT / f"sample_table_seed{SEED}_past{PAST_LEN}_future{FUTURE_LEN}.json"
split_path = SPLITS_ROOT / f"trajectory_split_seed{SEED}_past{PAST_LEN}_future{FUTURE_LEN}.json"

save_sample_table(sample_table, sample_table_path)
split_info = save_or_load_fixed_split(
    sample_table=sample_table,
    split_path=split_path,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    past_len=PAST_LEN,
    future_len=FUTURE_LEN,
)

summary = {
    "stream_count": len(streams),
    "sample_count": len(sample_table),
    "sample_table_path": str(sample_table_path),
    "split_path": str(split_path),
    "split_strategy": split_info["split_strategy"],
    "episode_count": split_info["episode_count"],
    "train_episode_ids": split_info["train_episode_ids"],
    "val_episode_ids": split_info["val_episode_ids"],
    "test_episode_ids": split_info["test_episode_ids"],
    "train_count": len(split_info["train_indices"]),
    "val_count": len(split_info["val_indices"]),
    "test_count": len(split_info["test_indices"]),
}
print(json.dumps(summary, indent=2))


{
  "stream_count": 6,
  "sample_count": 44541,
  "sample_table_path": "/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/train_ready/sample_table_seed42_past10_future5.json",
  "split_path": "/home/basudeo/Documents/Thesis/08_model_training_pipeline/results/splits/trajectory_split_seed42_past10_future5.json",
  "split_strategy": "episode",
  "episode_count": 6,
  "train_episode_ids": [
    "run_model_20260511_220103",
    "run_model_20260511_210809",
    "run_model_20260511_213251",
    "run_model_20260511_221726"
  ],
  "val_episode_ids": [
    "run_model_20260511_202309"
  ],
  "test_episode_ids": [
    "run_model_20260511_222947"
  ],
  "train_count": 29379,
  "val_count": 6195,
  "test_count": 8967
}


In [4]:
print("First sample:")
print(json.dumps(sample_table[0], indent=2)[:2000])

print("\nShared split preview:")
preview = {
    "train_sample_id": [split_info["sample_ids"][idx] for idx in split_info["train_indices"][:5]],
    "val_sample_id": [split_info["sample_ids"][idx] for idx in split_info["val_indices"][:5]],
    "test_sample_id": [split_info["sample_ids"][idx] for idx in split_info["test_indices"][:5]],
}
print(json.dumps(preview, indent=2))


First sample:
{
  "sample_id": "run_model_20260511_202309_stream000_start00000",
  "episode_id": "run_model_20260511_202309",
  "stream_index": 0,
  "start_index": 0,
  "anchor_index": 9,
  "past_len": 10,
  "future_len": 5,
  "anchor_timestamp_ns": 1778523801199261539,
  "teacher_state": "terrain",
  "future_xy_local": [
    [
      0.05359937545033749,
      -8.37885622467574e-05
    ],
    [
      0.07805100082799059,
      -0.000241164330703739
    ],
    [
      0.11402689628777356,
      -0.0005709491263258051
    ],
    [
      0.14210821164602164,
      -0.0008792957453454681
    ],
    [
      0.21048810494924947,
      -0.0016472451031522319
    ]
  ],
  "future_dt": [
    0.061451003000000004,
    0.14578307100000001,
    0.20208520600000002,
    0.24529497200000003,
    0.294628516
  ]
}

Shared split preview:
{
  "train_sample_id": [
    "run_model_20260511_210809_stream001_start00000",
    "run_model_20260511_210809_stream001_start00001",
    "run_model_20260511_210809_st